# Importation of necessary Python Libs

## Maths or data structure-related libraries

In [16]:
import math
import numpy as np
import random
import pandas as pd
from collections import deque, namedtuple
from re import match
from typing import Tuple, List, Set, Dict, Optional
from sklearn.linear_model import BayesianRidge

## Libraries for visualization/record tracking

In [3]:
import matplotlib.pyplot as plt
import tqdm

## Libraries for loading data

In [5]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os

Mounted at /content/drive


## Libraries for machine leanring

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler
# explicitly require this experimental feature
from sklearn.experimental import enable_iterative_imputer  # noqa
# now you can import normally from sklearn.impute
from sklearn.impute import IterativeImputer
from torch.utils.data import DataLoader, TensorDataset
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Implementation of the VAEQL algoritm

The link to my manuscript: <a>https://www.overleaf.com/project/67cb575babefcc1067d01469</a>

## Defining the miscelaneous methods

### Defining the training method depending on the hardware device

### Copy and paste the benchmark dataframe pre-processing methods from the previous experiments

In [7]:
def identify_binary_and_numerical_features(df: pd.DataFrame) -> Tuple[List[str]]:
     # 1. Identify “binary” columns (unique values ⊆ {0,1})
    categorical_cols = [col for col in df.columns if match(r".+-is_.+", col)]

    # 2. All the other numeric columns
    numeric_cols = list(set(df.columns) - set(categorical_cols))

    # return to a tuple of two lists of column names
    return numeric_cols, categorical_cols

In [8]:
def normalize_numerical_features(df: pd.DataFrame, num_features: List[str]) -> pd.DataFrame:
    # Create a MinMaxScaler instance
    scaler = MinMaxScaler()

    # Fit and transform the numerical features to the range [0, 1]
    df[num_features] = scaler.fit_transform(df[num_features])

    return df

In [9]:
def generate_masks_for_missingness(
    input_df: pd.DataFrame,
    original_df: pd.DataFrame,
    amputed_df: pd.DataFrame,
    num_feats: List[str],
    cat_feats: List[str],
) -> Tuple[np.ndarray, np.ndarray]:

    # Check if the dataframes match in shape
    if not original_df.shape == input_df.shape == amputed_df.shape:
        raise Exception("Sorry, the three dataframes do not match in shape")

    # Check if the dataframes have identical column names
    if not set(original_df.columns) == set(input_df.columns) == set(amputed_df.columns):
        raise Exception("Sorry, the three dataframes do not match in column names")

    # Initialize the maps for numerical and categorical features
    num_feat_map = np.zeros(original_df[num_feats].shape, dtype=int)
    cat_feat_map = np.zeros(original_df[cat_feats].shape, dtype=int)

    # Generate the missingness map for numerical features
    for i, feat in enumerate(num_feats):
        original_col = original_df[feat]
        amputed_col = amputed_df[feat]

        # 1: missing in the original dataset, regardless of whether it is amputed
        num_feat_map[:, i] = np.where(original_col.isna(), 1, 0)  # 1 if missing originally, otherwise 0
        # 2: existing in the original dataset and amputed
        num_feat_map[:, i] = np.where(amputed_col.isna(), 2, num_feat_map[:, i])  # 2 if amputed

    # Generate the missingness map for one-hot-encoded categorical features
    for i, feat in enumerate(cat_feats):
        original_col = original_df[feat]
        amputed_col = amputed_df[feat]

        # 1: missing in the original dataset, regardless of whether it is amputed
        cat_feat_map[:, i] = np.where(original_col.isna(), 1, 0)  # 1 if missing originally, otherwise 0
        # 2: existing in the original dataset and amputed
        cat_feat_map[:, i] = np.where(amputed_col.isna(), 2, cat_feat_map[:, i])  # 2 if amputed

    # Return the maps as a tuple of two arrays
    return num_feat_map, cat_feat_map

## Defining the Variational Autoencoder part of the VAEQL algorithm

### The VAE module

In [17]:
class MaskedVAE(nn.Module):
    def __init__(
        self,
        num_features: int,
        latent_dim: int | None = None,
        hidden_layer_sizes: tuple[int, ...] = (128, 64)
    ):
        super().__init__()
        self.num_features = num_features
        self.latent_dim = latent_dim or (num_features // 2)

        # Encoder layers
        enc_layers = []
        in_dim = num_features
        for h in hidden_layer_sizes:
            enc_layers += [nn.Linear(in_dim, h), nn.ReLU(inplace=True)]
            in_dim = h
        self.encoder = nn.Sequential(*enc_layers)
        self.fc_mu = nn.Linear(in_dim, self.latent_dim)
        self.fc_logvar = nn.Linear(in_dim, self.latent_dim)

        # Decoder layers
        dec_layers = []
        in_dim = self.latent_dim
        for h in reversed(hidden_layer_sizes):
            dec_layers += [nn.Linear(in_dim, h), nn.ReLU(inplace=True)]
            in_dim = h
        dec_layers += [nn.Linear(in_dim, num_features)]
        self.decoder = nn.Sequential(*dec_layers)

    def forward(self, x: torch.Tensor):
        # Encode
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        # Reparameterize
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        out_logits = self.decoder(z)
        return out_logits, mu, logvar

### The loss function

In [11]:
import torch.nn.functional as F

def masked_vae_loss(
    logits: torch.Tensor,
    target: torch.Tensor,
    mu: torch.Tensor,
    logvar: torch.Tensor,
    mask: torch.Tensor,
    beta: float = 1.0,
    eps: float = 1e-8
) -> torch.Tensor:
    """
    logits:  (batch, D)    raw decoder outputs
    target:  (batch, D)    original input in [0,1]
    mu:      (batch, Z)    latent mean
    logvar:  (batch, Z)    latent log‐variance
    mask:    (batch, D)    0=observed, 1=missing/original missing, 2=amputed
    """

    # 1) BCE on logits (handles sigmoid internally), no reduction
    bce_el = F.binary_cross_entropy_with_logits(
        logits, target, reduction='none'
    )

    # 2) Only originally observed entries contribute
    observed = (mask == 0).float()
    recon_loss = (bce_el * observed).sum() / (observed.sum() + eps)

    # 3) KL divergence term
    kl_loss = -0.5 * torch.sum(
        1 + logvar - mu.pow(2) - logvar.exp(),
        dim=1
    ).mean()

    # 4) Total loss
    return recon_loss + beta * kl_loss

### The VAE training method

In [12]:
def train_masked_vae(
    data_df: pd.DataFrame,
    mask_array: np.ndarray,
    export_prefix: str,
    hidden_layer_sizes: tuple[int, ...] = (128, 64),
    latent_dim: int | None = None,
    batch_size: int = 32,
    num_folds: int = 5,
    max_epochs: int = 100,
    learning_rate: float = 1e-3,
    beta: float = 1.0,
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
) -> list[float]:
    # Validate inputs
    if data_df.shape != mask_array.shape:
        raise ValueError("Data and mask shapes must match.")

    input_ts = torch.tensor(data_df.values, dtype=torch.float32)
    mask = torch.tensor(mask_array, dtype=torch.int64)
    num_samples, num_features = input_ts.shape
    latent_dim = latent_dim or (num_features // 2)

    kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
    fold_losses: list[float] = []
    last_model = None

    for fold, (train_idx, val_idx) in enumerate(kf.split(input_ts), start=1):
        model = MaskedVAE(num_features, latent_dim, hidden_layer_sizes).to(device)
        optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate)
        scheduler = ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)
        scaler = GradScaler()

        train_ds = TensorDataset(input_ts[train_idx], mask[train_idx])
        val_ds = TensorDataset(input_ts[val_idx], mask[val_idx])
        train_loader = DataLoader(train_ds, batch_size=batch_size,
                                  shuffle=True, num_workers=4, pin_memory=True)
        val_loader = DataLoader(val_ds, batch_size=batch_size,
                                shuffle=False, num_workers=2, pin_memory=True)

        best_val = float('inf')
        no_improve = 0
        train_loss_history: list[float] = []

        for epoch in range(1, max_epochs + 1):
            # training
            model.train()
            train_total = 0.0
            for xb, mb in train_loader:
                xb, mb = xb.to(device), mb.to(device)
                optimizer.zero_grad()
                with autocast():
                    logits, mu, logvar = model(xb, mb)
                    loss = masked_vae_loss(logits, xb, mu, logvar, mb, beta)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                train_total += loss.item() * xb.size(0)
            avg_train = train_total / len(train_idx)
            train_loss_history.append(avg_train)

            # validation
            model.eval()
            val_total = 0.0
            with torch.no_grad():
                for xb, mb in val_loader:
                    xb, mb = xb.to(device), mb.to(device)
                    logits, mu, logvar = model(xb, mb)
                    batch_loss = masked_vae_loss(logits, xb, mu, logvar, mb, beta)
                    val_total += batch_loss.item() * xb.size(0)
            avg_val = val_total / len(val_idx)
            scheduler.step(avg_val)

            # print per-epoch losses
            print(f"Fold {fold}, Epoch {epoch}: train={avg_train:.4f}, val={avg_val:.4f}")

            # early stopping
            if avg_val < best_val - 1e-4:
                best_val = avg_val
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= 10:
                    print(f"Fold {fold}: early stopping at epoch {epoch}")
                    break

        print(f"Fold {fold} final val loss: {best_val:.4f}")
        fold_losses.append(best_val)
        last_model = model

    # save only final fold model
    save_path = f"{export_prefix}.pth"
    torch.save(last_model.state_dict(), save_path)
    print(f"Saved final model to {save_path}")
    # export training loss history of last fold
    loss_df = pd.DataFrame({
        'epoch': list(range(1, len(train_loss_history)+1)),
        'train_loss': train_loss_history
    })
    loss_df.to_csv(f"{export_prefix}_train_loss.csv", index=False)
    print(f"Saved train loss history to {export_prefix}_train_loss.csv")

    return fold_losses

### Testing on the toy training method above

In [13]:
# create the list of tuples for pairing input
# 1. Point to the directory containing your CSVs
base_dir = "/content/drive/My Drive/VAEQL_paper"
amputed_dir = "/content/drive/My Drive/VAEQL_paper/amputed_datasets"
preprossed_dir = "/content/drive/My Drive/VAEQL_paper/preprocessed_datasets"

ref_dataset_dict = {
    "HCV_data": os.path.join(preprossed_dir, "HCV_data_unimputed.csv"),
    "appendicitis_data": os.path.join(preprossed_dir, "Regensburg_Pediatric_Appendicitis_unimputed.csv"),
    "Indian_liver_patients": os.path.join(preprossed_dir, "Indian_liver_patients_unimputed.csv"),
    "heart_failure_clinical_records": os.path.join(preprossed_dir, "Heart_Failure_Clinical_Records_complete.csv"),
    "HCV_Egyptian_patients": os.path.join(preprossed_dir, "HCV_Egyptian_data_complete.csv")
}

csv_path_pairs: List[Dict[str, str]] = []
for root, _, files in os.walk(amputed_dir):
    for f in files:
        if f.endswith(".csv"):
            amputed_path = os.path.join(root, f)
            dataset_name = root.split("/")[-1]
            brr_imputed_path = os.path.join(base_dir, "BRR_imputed_datasets", dataset_name, f)
            ref_dataset = os.path.join(preprossed_dir, ref_dataset_dict[dataset_name])
            csv_path_pairs.append({"amputed": amputed_path, "input": brr_imputed_path, "ref": ref_dataset})

print(f"Found {len(csv_path_pairs)} CSV file pairs in all subfolders.")

Found 500 CSV file pairs in all subfolders.


In [ ]:
csv_path_pairs[:3]

[{'amputed': '/content/drive/My Drive/VAEQL_paper/amputed_datasets/heart_failure_clinical_records/MAR_20_perc_1.csv',
  'input': '/content/drive/My Drive/VAEQL_paper/BRR_imputed_datasets/heart_failure_clinical_records/MAR_20_perc_1.csv',
  'ref': '/content/drive/My Drive/VAEQL_paper/preprocessed_datasets/Heart_Failure_Clinical_Records_complete.csv'},
 {'amputed': '/content/drive/My Drive/VAEQL_paper/amputed_datasets/heart_failure_clinical_records/MNAR_10_perc_1.csv',
  'input': '/content/drive/My Drive/VAEQL_paper/BRR_imputed_datasets/heart_failure_clinical_records/MNAR_10_perc_1.csv',
  'ref': '/content/drive/My Drive/VAEQL_paper/preprocessed_datasets/Heart_Failure_Clinical_Records_complete.csv'},
 {'amputed': '/content/drive/My Drive/VAEQL_paper/amputed_datasets/heart_failure_clinical_records/MAR_15_perc_1.csv',
  'input': '/content/drive/My Drive/VAEQL_paper/BRR_imputed_datasets/heart_failure_clinical_records/MAR_15_perc_1.csv',
  'ref': '/content/drive/My Drive/VAEQL_paper/preproce

In [ ]:
csv_path_pairs[-3:]

[{'amputed': '/content/drive/My Drive/VAEQL_paper/amputed_datasets/appendicitis_data/MNAR_15_perc_10.csv',
  'input': '/content/drive/My Drive/VAEQL_paper/BRR_imputed_datasets/appendicitis_data/MNAR_15_perc_10.csv',
  'ref': '/content/drive/My Drive/VAEQL_paper/preprocessed_datasets/Regensburg_Pediatric_Appendicitis_unimputed.csv'},
 {'amputed': '/content/drive/My Drive/VAEQL_paper/amputed_datasets/appendicitis_data/MNAR_20_perc_10.csv',
  'input': '/content/drive/My Drive/VAEQL_paper/BRR_imputed_datasets/appendicitis_data/MNAR_20_perc_10.csv',
  'ref': '/content/drive/My Drive/VAEQL_paper/preprocessed_datasets/Regensburg_Pediatric_Appendicitis_unimputed.csv'},
 {'amputed': '/content/drive/My Drive/VAEQL_paper/amputed_datasets/appendicitis_data/MNAR_25_perc_10.csv',
  'input': '/content/drive/My Drive/VAEQL_paper/BRR_imputed_datasets/appendicitis_data/MNAR_25_perc_10.csv',
  'ref': '/content/drive/My Drive/VAEQL_paper/preprocessed_datasets/Regensburg_Pediatric_Appendicitis_unimputed.cs

#### 1. <code>the Regensburg Pediatric Appendicitis Dataset<code> (large number of missing values, 713 patients)

##### Load and pre-process the data

In [ ]:
feat_m1, feat_c1 = identify_binary_and_numerical_features(pd.read_csv(csv_path_pairs[-2]["ref"]))

In [ ]:
test_apd_mask_num, test_apd_mask_cat = generate_masks_for_missingness(
    pd.read_csv(csv_path_pairs[-2]["input"]),
    pd.read_csv(csv_path_pairs[-2]["ref"]),
    pd.read_csv(csv_path_pairs[-2]["amputed"]),
    feat_m1,
    feat_c1
)

In [ ]:
input_df = pd.read_csv(csv_path_pairs[-2]["input"])
# This will set any value <0 to 0, and any value >1 to 1
input_df.clip(0, 1, inplace=True)

train_masked_vae(
    input_df,
    mask_array = np.hstack([test_apd_mask_num, test_apd_mask_cat])
)

/tmp/ipython-input-29-62664469.py:33: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipython-input-29-62664469.py:50: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Fold 1: early stopping at epoch 22
Fold 1 final val loss: 0.4918
Fold 2: early stopping at epoch 45
Fold 2 final val loss: 0.4950
Fold 3: early stopping at epoch 20
Fold 3 final val loss: 0.4879
Fold 4: early stopping at epoch 18
Fold 4 final val loss: 0.4787
Fold 5: early stopping at epoch 25
Fold 5 final val loss: 0.4708


[0.4918266730708676,
 0.4950475621890355,
 0.48791589353468034,
 0.4786739063934541,
 0.47084781443569024]

##### Apply pre-imputation by BRR to the amputed dataframes

#### 2. <code>the Heart Failure Clinical Records Dataset</code> (no missing values, 299 patients)

##### Load and pre-process the data

In [ ]:
csv_path_pairs[2]["ref"]

'/content/drive/My Drive/VAEQL_paper/preprocessed_datasets/Heart_Failure_Clinical_Records_complete.csv'

In [ ]:
feat_m2, feat_c2 = identify_binary_and_numerical_features(pd.read_csv(csv_path_pairs[2]["ref"]))

In [ ]:
test_hf_mask_num, test_hf_mask_cat = generate_masks_for_missingness(
    pd.read_csv(csv_path_pairs[2]["input"]),
    pd.read_csv(csv_path_pairs[2]["ref"]),
    pd.read_csv(csv_path_pairs[2]["amputed"]),
    feat_m2,
    feat_c2
)

In [ ]:
input_df = pd.read_csv(csv_path_pairs[2]["input"])
# This will set any value <0 to 0, and any value >1 to 1
input_df.clip(0, 1, inplace=True)

train_masked_vae(
    input_df,
    mask_array = np.hstack([test_hf_mask_num, test_hf_mask_cat])
)

/tmp/ipython-input-29-62664469.py:33: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipython-input-29-62664469.py:50: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Fold 1: early stopping at epoch 22
Fold 1 final val loss: 0.6020
Fold 2: early stopping at epoch 12
Fold 2 final val loss: 0.6095
Fold 3: early stopping at epoch 33
Fold 3 final val loss: 0.6024
Fold 4: early stopping at epoch 28
Fold 4 final val loss: 0.6097
Fold 5: early stopping at epoch 27
Fold 5 final val loss: 0.6322


[0.6019824504852295,
 0.6095417539278666,
 0.6023689190546672,
 0.609739875793457,
 0.6322469943660801]

##### Do the pre-training of the VAEs on GPU and export the VAE models of each dataset

In [ ]:
for csv_path_pair in csv_path_pairs:

    feat_m, feat_c = identify_binary_and_numerical_features(pd.read_csv(csv_path_pair["ref"]))

    mask_num, mask_cat = generate_masks_for_missingness(
      pd.read_csv(csv_path_pair["input"]),
      pd.read_csv(csv_path_pair["ref"]),
      pd.read_csv(csv_path_pair["amputed"]),
      feat_m,
      feat_c
    )

    input_df = pd.read_csv(csv_path_pair["input"])
    # This will set any value <0 to 0, and any value >1 to 1
    input_df.clip(0, 1, inplace=True)

    train_masked_vae(
        input_df,
        np.hstack([mask_num, mask_cat]),
        os.path.join("/content/drive/My Drive/VAEQL_paper/trained_VAEs", csv_path_pair["input"].split("/")[-2], csv_path_pair["input"].split("/")[-1].split(".")[0])
    )

## Defining the Vanilla Q-learning and VAE-update part of the VAEQL algorithm

In [42]:
class QImputer:
    def __init__(
        self,
        ref_df: pd.DataFrame,
        mask: np.ndarray,                       # shape (N, D), values ∈ {0,1,2}
        num_feats: List[str],
        vae_pth_path: str,
        latent_dim: Optional[int] = None,
        hidden_layer_sizes: Tuple[int, ...] = (128,64),
        alpha: float   = 0.1,
        gamma: float   = 0.99,
        epsilon: float = 0.1,
        zeta1: float = 0.5,
        zeta2: float = 1,
        device: str    = "cuda",
    ):
        """
        QImputer: off‑policy Q‑learning to refine type‑2 (amputed) imputations via a pretrained MaskedVAE.
        """
        # --- 0) Device ---
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")

        # --- 1) Load data + mask as tensors ---
        data_np      = ref_df.values.astype(np.float32)
        self.data_t  = torch.from_numpy(data_np).to(self.device)    # (N, D)
        self.mask_t       = torch.from_numpy(mask).long().to(self.device) # (N, D)
        self.obs_mask = (self.mask_t == 0)  # torch.BoolTensor
        self.N, self.D = self.data_t.shape

        # Flattened view for easy indexing
        self.data_flat_t = self.data_t.reshape(-1)  # (N*D,)

        # --- 2) Build idx0 (observed) and idx2 (amputed numeric) as LongTensors ---
        flat_mask = self.mask_t.reshape(-1)              # (N*D,)
        self.idx0_t = (flat_mask == 0).nonzero(as_tuple=False).squeeze(1)

        # Identify your numerical columns
        cat_cols = [c for c in ref_df.columns if match(r".+-is_.+", c)]
        num_cols = [c for c in ref_df.columns if c not in cat_cols]
        col_to_idx = {c:i for i,c in enumerate(ref_df.columns)}
        num_idx = torch.tensor([col_to_idx[c] for c in num_feats],
                               dtype=torch.long, device=self.device)

        all_idx = torch.arange(self.N*self.D, device=self.device)
        col_idx = all_idx % self.D
        cond2   = (flat_mask == 2)
        condNum = torch.isin(col_idx, num_idx)
        self.idx2_t = all_idx[cond2 & condNum]   # (M,)
        self.M      = self.idx2_t.numel() # number of type 2 numerical elements

        # BRR baseline values for those M positions
        self.brr_baseline_t = self.data_flat_t[self.idx2_t].clone()  # (M,)

        # --- 3) Hyperparams & Q‑table as tensor ---
        self.alpha   = alpha
        self.gamma   = gamma
        self.epsilon = epsilon
        self.zeta1    = zeta1
        self.zeta2    = zeta2
        # Q shape = (M, 101 states, 3 actions)
        self.Q = torch.randn(self.M, 101, 3, device=self.device)

        # --- 4) Load pretrained VAE ---
        self.vae = MaskedVAE(
            num_features=self.D,
            latent_dim=latent_dim,
            hidden_layer_sizes=hidden_layer_sizes
        ).to(self.device)
        self.vae.load_state_dict(torch.load(f"{vae_pth_path}.pth", map_location=self.device))


    @staticmethod
    def _discretize_torch(vals: torch.Tensor) -> torch.LongTensor:
        """ Round [0,1]→{0,…,100} """
        return (vals * 100.0).round().clamp(0,100).long()


    def train(
        self,
        max_iters:   int   = 1000,
        conv_thresh: float = 1e-3,
    ) -> torch.Tensor:
        """
        Run Q‑learning until convergence or max_iters.
        Returns:
            best_tensor: FloatTensor of shape (N, D) with refined imputations.
        """
        # 1) init best candidate & helpers
        best_tensor = self.data_t.clone()       # (N, D)
        best_reward = -float("inf")
        arange_M    = torch.arange(self.M, device=self.device)

        for it in range(max_iters):
            # 2) reconstruct current best
            with torch.no_grad():
                rec, _, _ = self.vae(best_tensor)  # (N, D)
            rec_flat = rec.reshape(-1) # (N*D,)

            # 3) extract M current values & discretize to states
            curr_vals = rec_flat[self.idx2_t]                  # (M,)
            current_states = self._discretize_torch(curr_vals)      # LongTensor (M,)

            # 4) ε‑greedy action selection
            if torch.rand(1, device=self.device).item() < self.epsilon:
                actions = torch.randint(0,3,(self.M,), device=self.device)
            else:
                # pick action with highest Q for each state
                actions = self.Q[arange_M, current_states].argmax(dim=1)     # (M,)

            # 5) apply actions → next_vals
            next_vals = (curr_vals + (actions - 1)/100.0).clamp(0,1)  # (M,)

            # 6) build candidate tensor
            cand = best_tensor.clone()
            cand_flat = cand.reshape(-1)
            cand_flat[self.idx2_t] = next_vals
            cand = cand_flat.view(self.N, self.D)

            # 7) reconstruct candidate
            with torch.no_grad():
                rec2, _, _ = self.vae(cand)
            rec2_flat = rec2.reshape(-1)

            # 8) compute the global reward
            old_err0 = (rec_flat[self.idx0_t] - self.data_flat_t[self.idx0_t]).pow(2).mean()
            new_err0 = (rec2_flat[self.idx0_t] - self.data_flat_t[self.idx0_t]).pow(2).mean()
            delta_err0   = old_err0 - new_err0

            delta_err2   = ((curr_vals - self.brr_baseline_t).pow(2) -
                    (next_vals - self.brr_baseline_t).pow(2)).mean()
            global_reward = delta_err0 + self.zeta1 * delta_err2

            # 9) Q‑learning update (track max abs update)
            next_states = self._discretize_torch(next_vals)
            best_nextQ = self.Q[arange_M, next_states].max(dim=1)[0]  # (M,)

            max_delta = 0.0
            for i in range(self.M):
                state  = current_states[i]
                action = actions[i]
                # --- NEW: individual reward for the same patient ---
                idx = self.idx2_t[i].item()           # flattened index
                row = idx // self.D                   # which patient
                # pick that patient's observed columns:
                obs_cols = self.obs_mask[row]              # Boolean vector length D
                old_vals = rec[row, obs_cols]     # 1‐d tensor of that patient’s observed reconstructions
                new_vals = rec2[row, obs_cols]
                true_vals= self.data_t[row, obs_cols]  # ground truth at those spots

                # compute change in MSE for that patient:
                old_loss_row = (old_vals - true_vals).pow(2).mean()
                new_loss_row = (new_vals - true_vals).pow(2).mean()
                indiv_reward = old_loss_row - new_loss_row

                td_target  = global_reward + self.zeta2 * indiv_reward + self.gamma * best_nextQ[i]
                td  = td_target - self.Q[i, state, action]
                q_delta     = self.alpha * td
                self.Q[i, state, action] += q_delta
                max_delta = max(max_delta, q_delta.abs().item())

            # 10) track best candidate
            if global_reward.item() > best_reward:
                best_reward = global_reward.item()
                best_tensor = cand.clone()

            # 11) convergence check
            if max_delta < conv_thresh:
                print(f"Converged at iter {it}, maxΔQ={max_delta:.2e}")
                break
            if it % 100 == 0:
                print(f"Iter {it:4d}: R={global_reward.item():.4f}  bestR={best_reward:.4f}  maxΔQ={max_delta:.2e}")

        return best_tensor

# Testing the trainer class above on the five pre-processed datasets

In [26]:
csv_path_pairs[-5:]

[{'amputed': '/content/drive/My Drive/VAEQL_paper/amputed_datasets/appendicitis_data/MNAR_5_perc_10.csv',
  'input': '/content/drive/My Drive/VAEQL_paper/BRR_imputed_datasets/appendicitis_data/MNAR_5_perc_10.csv',
  'ref': '/content/drive/My Drive/VAEQL_paper/preprocessed_datasets/Regensburg_Pediatric_Appendicitis_unimputed.csv'},
 {'amputed': '/content/drive/My Drive/VAEQL_paper/amputed_datasets/appendicitis_data/MNAR_10_perc_10.csv',
  'input': '/content/drive/My Drive/VAEQL_paper/BRR_imputed_datasets/appendicitis_data/MNAR_10_perc_10.csv',
  'ref': '/content/drive/My Drive/VAEQL_paper/preprocessed_datasets/Regensburg_Pediatric_Appendicitis_unimputed.csv'},
 {'amputed': '/content/drive/My Drive/VAEQL_paper/amputed_datasets/appendicitis_data/MNAR_15_perc_10.csv',
  'input': '/content/drive/My Drive/VAEQL_paper/BRR_imputed_datasets/appendicitis_data/MNAR_15_perc_10.csv',
  'ref': '/content/drive/My Drive/VAEQL_paper/preprocessed_datasets/Regensburg_Pediatric_Appendicitis_unimputed.csv'

In [29]:
MAX_QL_ITER = 10

In [ ]:
for csv_path_pair in csv_path_pairs[-5:]:

    prev_file_suffix = csv_path_pair["input"].split("/")[-1].split(".")[0]
    pre_trained_VAE_pth_path = os.path.join("/content/drive/My Drive/VAEQL_paper/trained_VAEs", csv_path_pair["input"].split("/")[-2], prev_file_suffix)

    feat_m, feat_c = identify_binary_and_numerical_features(pd.read_csv(csv_path_pair["ref"]))

    mask_num, mask_cat = generate_masks_for_missingness(
      pd.read_csv(csv_path_pair["input"]),
      pd.read_csv(csv_path_pair["ref"]),
      pd.read_csv(csv_path_pair["amputed"]),
      feat_m,
      feat_c
    )

    input_df = pd.read_csv(csv_path_pair["input"])
    # This will set any value <0 to 0, and any value >1 to 1
    input_df.clip(0, 1, inplace=True)

    for ql_iter in range(MAX_QL_ITER):

      last_VAE_pth_path = f"{prev_file_suffix}-QL_iter_{ql_iter-1}" if ql_iter > 0 else pre_trained_VAE_pth_path
      new_file_predix = f"{prev_file_suffix}-QL_iter_{ql_iter}"

      q_imputer = QImputer(ref_df = input_df, mask = np.hstack([mask_num, mask_cat]), num_feats=feat_m,
        vae_pth_path = last_VAE_pth_path,
        gamma = 0.95,
        epsilon = 0.1,
        zeta1 = 0.25,
        zeta2 = 2
      )

      q_learnt_tensor = q_imputer.train()

      train_masked_vae(
          q_learnt_tensor,
          np.hstack([mask_num, mask_cat]),
          os.path.join("/content/drive/My Drive/VAEQL_paper/trained_VAEs", csv_path_pair["input"].split("/")[-2], new_file_predix)
      )

Iter    0: R=0.1209  bestR=0.1209  maxΔQ=1.07e+00
